# September: SigLIP embeddings on OmniAI OCR Benchmark
Embeds all 1,000 documents with **SigLIP**, caches to disk incrementally (resumable), then evaluates with **three clustering algorithms** (k-means, agglomerative, GMM) plus a quick supervised check. No OCR. Embeddings don't depend on labels, so we embed everything and decide drop/bucket for the 36 classes later.

Default checkpoint: `google/siglip-base-patch16-224` (swap for `siglip-so400m-patch14-384` in the model cell to test higher resolution).
Run in order. **Restart the runtime after the install cell**, then run from cell 2. This notebook is self-contained; the other model has its own notebook, and both write to the same cache folder.

## 1. Setup

In [ ]:
!pip -q install -U transformers datasets peft accelerate einops umap-learn scikit-learn
import transformers, torch, sys
print("python", sys.version.split()[0], "| transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")
# Copy these version numbers into the README (project requires pinned versions).

In [ ]:
import os, json, numpy as np, pandas as pd, torch
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Where to cache. Colab: shared Drive folder. Kaggle: /kaggle/working. ---
if os.path.exists("/kaggle/working"):
    CACHE = Path("/kaggle/working/embeddings")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE = Path("/content/drive/MyDrive/legal_idp/embeddings")   # share this folder with your teammate
    except ImportError:
        CACHE = Path("./embeddings")
CACHE.mkdir(parents=True, exist_ok=True)
print("Cache dir:", CACHE)

## 2. Data
Upload `labels.csv` (from `load_data.py`) next to the notebook, or put it in the cache folder.

In [ ]:
from datasets import load_dataset

labels_path = next(p for p in [Path("labels.csv"), CACHE / "labels.csv", Path("/kaggle/input/labels/labels.csv")] if p.exists())
labels = pd.read_csv(labels_path)

ds = load_dataset("getomni-ai/ocr-benchmark", split="test")
ds = ds.select_columns(["id", "image"])          # skip the big text columns
assert list(ds["id"]) == labels["id"].tolist(), "labels.csv order must match dataset order"
print(len(ds), "documents;", labels["format"].nunique(), "format labels")
np.save(CACHE / "ids.npy", labels["id"].to_numpy())
labels.to_csv(CACHE / "labels.csv", index=False)

## 3. Incremental caching helper
Writes a shard every few batches and **resumes** where it stopped if the session dies.

In [ ]:
def to_rgb(img, max_side=None, pad_square=False):
    img = img.convert("RGB")
    if max_side and max(img.size) > max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    if pad_square:                                   # keep aspect ratio, white padding (same idea as the EDA notebook)
        s = max(img.size); img = ImageOps.pad(img, (s, s), color="white")
    return img

def embed_all(name, encode_fn, batch_size, shard_every=10, **img_kw):
    """encode_fn(list[PIL]) -> np.ndarray [B, D]. Saves shards to CACHE/name_shards/, then merges to CACHE/name.npy"""
    final = CACHE / f"{name}.npy"
    if final.exists():
        print("already done:", final); return np.load(final)
    sd = CACHE / f"{name}_shards"; sd.mkdir(exist_ok=True)
    step = batch_size * shard_every
    for start in tqdm(range(0, len(ds), step), desc=name):
        shard = sd / f"{start:05d}.npy"
        if shard.exists(): continue                   # resume
        vecs = []
        for b in range(start, min(start + step, len(ds)), batch_size):
            imgs = [to_rgb(ds[i]["image"], **img_kw) for i in range(b, min(b + batch_size, len(ds), start + step))]
            with torch.inference_mode():
                vecs.append(encode_fn(imgs))
        np.save(shard, np.concatenate(vecs))
    emb = np.concatenate([np.load(p) for p in sorted(sd.glob("*.npy"))]).astype("float32")
    assert emb.shape[0] == len(ds) and np.isfinite(emb).all(), f"bad embeddings {emb.shape}"
    np.save(final, emb); print(name, emb.shape); return emb

## 4. SigLIP
Default `google/siglip-base-patch16-224` (fast baseline). For a fairer shot at documents try a higher-res checkpoint (`google/siglip-so400m-patch14-384`). Note: SigLIP resizes to a fixed small square, so fine text is mostly lost; expect it to separate by *layout*, not content. That is itself a finding for the write-up.

In [ ]:
from transformers import AutoModel, AutoProcessor

SIGLIP_ID = "google/siglip-base-patch16-224"     # try: "google/siglip-so400m-patch14-384"
sig_model = AutoModel.from_pretrained(SIGLIP_ID, torch_dtype=torch.float16).to(device).eval()
sig_proc = AutoProcessor.from_pretrained(SIGLIP_ID)

def encode_siglip(imgs):
    px = sig_proc(images=imgs, return_tensors="pt")["pixel_values"].to(device, torch.float16)
    f = sig_model.get_image_features(pixel_values=px)
    if not torch.is_tensor(f): f = f.pooler_output
    f = torch.nn.functional.normalize(f.float(), dim=-1)
    return f.cpu().numpy()

name = "siglip_" + SIGLIP_ID.split("/")[-1]
emb_siglip = embed_all(name, encode_siglip, batch_size=32, pad_square=True)
del sig_model; torch.cuda.empty_cache()

## Evaluation
Head classes only (the 18 with 30+ docs) for metrics that need enough examples per class. Fixed seed, 5-fold stratified CV. This is a first look, not the final split.
Clustering: **k-means**, **agglomerative (ward)**, **GMM** (fit on 50 PCA dims, since a full-covariance GMM on 768/2048 dims with ~925 points is ill-conditioned). k = number of true head classes. Metrics: ARI, NMI, silhouette (computed on the true labels and on each clustering).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (f1_score, accuracy_score, adjusted_rand_score,
                             normalized_mutual_info_score, silhouette_score, classification_report)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

PREFIX = "siglip_"
labels = pd.read_csv(CACHE / "labels.csv")
head = labels["is_head_class"].to_numpy()
y = labels.loc[head, "format"].to_numpy()
k = len(np.unique(y))
files = sorted(p for p in CACHE.glob(PREFIX + "*.npy") if "shards" not in p.name)
print("evaluating:", [p.name for p in files])

def cluster_scores(Xh, y, k):
    Xp = PCA(50, random_state=SEED).fit_transform(Xh)
    algos = {
        "kmeans": KMeans(k, n_init=10, random_state=SEED).fit_predict(Xh),
        "agglomerative": AgglomerativeClustering(k, linkage="ward").fit_predict(Xh),
        "gmm": GaussianMixture(k, covariance_type="diag", n_init=3, random_state=SEED).fit(Xp).predict(Xp),
    }
    out = []
    for name, c in algos.items():
        out.append({"algorithm": name, "ARI": adjusted_rand_score(y, c), "NMI": normalized_mutual_info_score(y, c),
                    "silhouette(pred clusters)": silhouette_score(Xh, c, metric="cosine")})
    return pd.DataFrame(out)

all_rows = []
for p in files:
    X = np.load(p); X = X / np.linalg.norm(X, axis=1, keepdims=True); Xh = X[head]
    cs = cluster_scores(Xh, y, k)
    cs.insert(0, "model", p.stem)
    cs["silhouette(true labels)"] = silhouette_score(Xh, y, metric="cosine")
    pred = cross_val_predict(LogisticRegression(max_iter=2000, C=10), Xh, y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    cs["LR macroF1"] = f1_score(y, pred, average="macro"); cs["LR acc"] = accuracy_score(y, pred)
    all_rows.append(cs)
    print("=====", p.stem, "per-class report"); print(classification_report(y, pred, digits=2))
results = pd.concat(all_rows).round(3).set_index(["model", "algorithm"])
results.to_csv(CACHE / f"{PREFIX}quick_results.csv"); results

In [ ]:
# UMAP colored by class (all 1000 docs; tail classes unlabeled in legend)
import umap, matplotlib.pyplot as plt
for p in files:
    X = np.load(p)
    Z = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(X)
    fig, ax = plt.subplots(figsize=(9, 7))
    for f, g in labels.groupby("format"):
        ax.scatter(*Z[g.index].T, s=10, label=f if len(g) >= 30 else None, alpha=.7)
    ax.set_title(p.stem); ax.legend(fontsize=6, ncol=2); plt.show()